# MLSys-2026 一键跑分（Colab / Kaggle 免费 GPU）

本机是 Intel Arc，跑不了 CUDA / Triton，所以用云上的免费 NVIDIA GPU 来产出实测数据。

**用法：**
1. 菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`
2. 在下面第 3 个单元格里把 `REPO_URL` 改成你已经推上 GitHub 的仓库地址
3. 从上到下依次运行（或 `Ctrl+F9` 全部运行）
4. 最后一步会打包 `results/`，顺手把 zip 下载下来，贴回 task1 / task2 report.md 的 [待填] 位置

> 注意：Colab 免费层用 T4（sm_75），`-arch=native` 会正确指向 sm_75；脚本里已做兜底。

In [ ]:
# 1. 环境确认：必须看到 NVIDIA GPU
!nvidia-smi!nvcc --versionimport sys, torchprint('python :', sys.version.split()[0])print('torch  :', torch.__version__, '| cuda:', torch.version.cuda, '| available:', torch.cuda.is_available())print('device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')try:    import triton    print('triton :', triton.__version__)except Exception as exc:    print('triton : not installed ->', exc)

In [ ]:
# 2. 拉取仓库（把下面的地址改成你自己的仓库）
REPO_URL = 'https://github.com/USERNAME/MLSys-2026.git'import osif os.path.isdir('MLSys-2026/.git'):    !cd MLSys-2026 && git pullelse:    !git clone {REPO_URL} MLSys-2026%cd MLSys-2026!ls

In [ ]:
# 3. Task 1：编译全部 .cu，跑完 Vector Add 与三个 GEMM 变体
#    产物：task1/results/*.csv 与 environment.txt（自动记录 nvidia-smi / nvcc / 编译参数）
!bash task1/benchmark/build_and_run.sh

In [ ]:
# 4. Task 1：画图（Shape / Tile Size 对性能的影响）
!pip -q install pandas matplotlib!python task1/benchmark/plot_results.pyimport globfrom IPython.display import Image, displayfor p in sorted(glob.glob('task1/results/figures/*.png')):    print(p)    display(Image(p))

In [ ]:
# 5. Task 2：Triton Vector Add / GEMM / RMSNorm / 72 配置 Autotuning
#    Colab 一般自带 triton；若报 ModuleNotFoundError，把下面这行的注释去掉再跑
# !pip -q install triton%cd task2!python triton/bench_all.py

In [ ]:
# 6. 打包 results 并下载回本机
%cd ..import shutilshutil.make_archive('mlsys_results_t1', 'zip', root_dir='.', base_dir='task1/results')shutil.make_archive('mlsys_results_t2', 'zip', root_dir='.', base_dir='task2/results')from google.colab import filesfiles.download('mlsys_results_t1.zip')files.download('mlsys_results_t2.zip')

## 拿到数据之后

把 `environment.txt` 和 CSV 发我，我帮你把 `task1/report.md`、`task2/report.md` 里所有
**[待填]** 的表格补齐，并按报告里已写好的框架补上基于实测的解释：

- Block Size 增大后吞吐为何不再上升、1024 是否反而更慢
- TILE 8 / 16 / 32 与方阵 / 小 M Shape 的性能差异
- Register Tiling 的收益与 Register Pressure 的代价
- Autotune 最佳配置在 held-out Shape 上的迁移性